# T30C / E13 — Sailor2-8B: đối chiếu `float16` với kích hoạt `float32`

Lượt T30B (24/11/2026) cho độ lệch `float16`–`bfloat16` là 0,025 với Qwen2.5-7B và 0,069 với Sailor2-8B. Nhưng T07
đo Qwen `float16` chỉ lệch `float32` 0,0007, tức 0,025 gần như là sai số của chính `bfloat16`. `bfloat16` vì vậy
không đủ chính xác để làm đối chiếu.

**Cách làm mới.** Nạp mô hình như `float16` (vừa bộ nhớ), rồi chuyển mọi tensor không phải trọng số 4 bit sang
`float32` và cho các lớp 4 bit tính ở `float32`. Qwen2.5-7B chạy cùng cách để kiểm chính phép đo: nếu cho lại
cỡ 0,0007 thì cách này dùng được.

**Dự đoán ghi trước khi chạy:** (1) Qwen2.5-7B lệch dưới 0,005; (2) Sailor2-8B lệch dưới 0,01.

Ước tính 15–20 phút T4.

In [ ]:
# Ô 1 — lấy mã nguồn. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

In [ ]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.data.paths import find_raw_dir

problems = []
absent = [n for n in ("torch", "transformers", "bitsandbytes", "accelerate", "pandas")
          if importlib.util.find_spec(n) is None]
if absent:
    problems.append(f"thiếu gói {absent}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
else:
    print("GPU:", torch.cuda.get_device_name(0))
try:
    raw = Path(find_raw_dir())
    missing = [n for n in ['vihallu_train.csv', 'isedsc01_train.json'] if not (raw / n).is_file()]
    print("dữ liệu thô:", raw, "— thiếu", missing or "không")
    if missing:
        problems.append(f"thiếu tệp {missing}")
except Exception as error:
    problems.append(f"chưa gắn dataset dữ liệu thô ({error})")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

In [ ]:
# Ô 4 — kiểm môi trường.
!python scripts/probe_env.py
!python -m pytest tests/test_compare_dtypes.py tests/test_revisions.py -q

In [ ]:
# Ô 5 — mỗi lượt một tiến trình. Một lượt hỏng KHÔNG dừng các lượt sau.
# Lượt float32_late hết bộ nhớ thì tự thử lại với ngữ cảnh tối đa 3.072 rồi 2.048 token; lượt
# float16 chạy ở đúng giới hạn đó để hai lượt so được với nhau.
import os
import sys
from pathlib import Path

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
OUT = Path("/kaggle/working/t30c")
OUT.mkdir(exist_ok=True)
MODELS = {"qwen7b": "Qwen/Qwen2.5-7B-Instruct", "sailor2": "sail/Sailor2-8B-SFT"}


def one(key, dtype, limit):
    target = OUT / f"{key}_{dtype}_{limit}.npz"
    if target.is_file():
        return True
    print("=" * 78)
    print(f"  {MODELS[key]} ở {dtype}, tối đa {limit} token", flush=True)
    done = subprocess.run([sys.executable, "scripts/compare_dtypes.py", "--model", MODELS[key],
                           "--dtype", dtype, "--max-context-tokens", str(limit),
                           "--save", str(target)], capture_output=True, text=True)
    print(done.stdout[-1200:])
    if done.returncode:
        print(done.stderr[-2000:])
    return target.is_file()


for key in MODELS:
    for limit in (4096, 3072, 2048):
        if one(key, "float32_late", limit):
            one(key, "float16", limit)
            break

In [ ]:
# Ô 6 — so sánh.
for high in sorted(OUT.glob("*_float32_late_*.npz")):
    low = OUT / high.name.replace("float32_late", "float16")
    if not low.is_file():
        print(f"{high.name}: thiếu lượt float16")
        continue
    done = subprocess.run([sys.executable, "scripts/compare_dtypes.py", "--from-saved",
                           str(low), str(high)], capture_output=True, text=True)
    print("=" * 78)
    print(done.stdout or done.stderr)
    (OUT / high.name.replace(".npz", "_so_sanh.txt")).write_text(done.stdout, encoding="utf-8")

In [ ]:
# Ô 7 — gói lại để tải về: /kaggle/working/t30c.zip
import shutil

print(shutil.make_archive("/kaggle/working/t30c", "zip", "/kaggle/working/t30c"))